# 02b · E1 error analysis (exploratory)

Why do models get items right in English but wrong in Hausa? This notebook codes 50 such
**discordant** items (correct in `en_p0`, wrong in `ha_p0`): 25 from Llama-3.2-3B and 25 from
MedGemma-4B, drawn with the fixed seed in `src/config.py` (deviation 4). No hypothesis test
is attached.

**This is the second coding pass (deviation 5).** The first pass is superseded: 20 of its 29
text-error rows named a segment that cannot carry the error they claim, and no row had a note.
The sample is unchanged — the same 50 rows are coded again, under written definitions, by two
coders independently.

- **Part A** exports two identical blank sheets, one per coder.
- **You and the second coder** code them independently, following `data/e1_codebook.md`.
- **Part B** validates both, reports agreement, adjudicates disagreements, writes `e1_summary.json`.

No GPU and no Drive needed: every step works from the sheet alone.

## 0 · Setup

In [ ]:
REPO_DIR = "/content/hausa-med-qa"
import os
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/avvas200/hausa-med-qa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log -1 --format="%h %s"

In [ ]:
import json
import pandas as pd
from src import config as C, data as D, e1 as E1

# The first pass is kept only as the source of the 50 rows' text; its codes are not used.
V1 = "results/e1_coding_v1_superseded.csv"
KEY = "results/e1_key.csv"          # unchanged: same sample, same rows
src_sheet = E1.read_sheet(V1)
print(len(src_sheet), "rows carried over from the superseded sheet")
print("codebook:", ", ".join(E1.CODES))

# Part A · Export the two coder sheets

In [ ]:
blank = E1.blank_sheet(src_sheet)
paths = {n: f"data/e1_coding_coder{n}.csv" for n in (1, 2)}
for n, p in paths.items():
    if os.path.exists(p):
        # Export is deterministic, so a re-run must reproduce the committed sheet exactly.
        old = E1.read_sheet(p)
        assert old.equals(blank.astype(str)), f"{p} differs from a fresh export - stop and check"
        print(f"{p} already exported and matches a fresh export")
    else:
        blank.to_csv(p, index=False, encoding="utf-8-sig")
        print(f"wrote {p}: {len(blank)} rows")

print("
rows with >=1 option identical in EN and HA:",
      int((blank.n_identical_options.astype(int) > 0).sum()), "of", len(blank))
print("rows where no answer letter could be parsed:", int((blank.unparseable == "yes").sum()))

### How to code

Read **[`data/e1_codebook.md`](../data/e1_codebook.md)** first — it holds the written definitions,
generated from `CODEBOOK` in `src/e1.py` so the two cannot drift apart.

Open your sheet in Google Sheets and fill three columns on every row: `code`, `segment`, `notes`.
Check the four categories **in order** and use the first that applies: `format`, `term`,
`translation`, `reasoning`.

The sheet is validated on read and will be **rejected** with the offending rows named if:

| | |
|---|---|
| R1 | `code` is not one of the four |
| R2 | `notes` is blank or under 10 characters — **every row needs one** |
| R3 | `term`/`translation` have no `segment`, or `reasoning` has one |
| R4 | the coded `segment` is identical in English and Hausa — see `identical_options` |
| R5 | a `term`/`translation` segment is neither gold nor the model's choice — see `codeable_segments` |

R4 and R5 are what the first pass got wrong, so the columns that pre-compute them
(`identical_options`, `codeable_segments`) are on the sheet for you.

Two further points:
- **Do not code `format` because no letter was parsed.** That is the `unparseable` column, set
  automatically, and it is 0 on all 50 rows here.
- **Do not open `results/e1_key.csv`.** It names the model behind each row. (It is public, so the
  author's pass is unblinded by necessity — the second coder's must not be.)

Save as CSV to `data/e1_coding_coder1.csv` (author) and `data/e1_coding_coder2.csv` (second coder).

# Part B · Validate, measure agreement, adjudicate

In [ ]:
merged, agree, disagreements = E1.merge_coders(paths[1], paths[2])
print(f"raw agreement {agree['raw_agreement']:.3f}   Cohen's kappa {agree['cohen_kappa']:.3f}"
      f"   (n={agree['n']})")
print("
confusion (rows = coder 1, cols = coder 2):")
print(pd.DataFrame(agree["confusion"]).fillna(0).astype(int).to_string())
print(f"
{len(disagreements)} row(s) need adjudication")

ADJ = "data/e1_adjudication.csv"
if len(disagreements) and not os.path.exists(ADJ):
    disagreements.to_csv(ADJ, index=False, encoding="utf-8-sig")
    print("wrote", ADJ, "- settle each row together, filling code/segment/notes")

### Adjudicating

`data/e1_adjudication.csv` shows both coders' `code`, `segment` and `notes` side by side
(`_1` and `_2`) with blank `code`/`segment`/`notes` columns at the end. Settle each row together
and fill those three. The same five rules apply, and the final codes are validated again before
anything is written.

Agreement is reported **before** adjudication, which is the figure that goes in the report.

In [ ]:
coded = E1.finalise(merged, ADJ if len(disagreements) else None)
coded = coded.merge(E1.read_sheet(KEY), on="row")
print(pd.crosstab(coded.model, coded.code, margins=True).to_string())

summary = E1.summarise(coded, agreement_stats=agree)
summary["sha256"] = {f: D.sha256(f) for f in
                     [paths[1], paths[2], KEY] + ([ADJ] if len(disagreements) else [])}

p = summary["primary_pooled"]
print(f"
== POOLED, n={p['n']} (primary) ==")
for c, v in p["codes"].items():
    print(f"  {c:12s} {v['n']:3d}  {v['prop']:.2f}  95% CI {v['ci95'][0]:.2f}-{v['ci95'][1]:.2f}")
print("  text errors sit in:", p["text_error_location"])
print("  status of those segments:", p["text_error_segment_status"])

print("
== per model (descriptive only - the author coded unblinded) ==")
for m, s in summary["descriptive_per_model"].items():
    print(f"  {m:10s}", {c: v["n"] for c, v in s["codes"].items()})

json.dump(summary, open("results/e1_summary.json", "w"), indent=2)
print("
wrote results/e1_summary.json")

## Next

Commit `data/e1_coding_coder1.csv`, `data/e1_coding_coder2.csv`,
`data/e1_adjudication.csv` (if any) and `results/e1_summary.json`,
then replace the "being re-coded" block in `README.md` with the pooled figures above.